# `cavanaulton/Pose2Par`: reproducibility and validation audit

**Classification: Level C — source review only.** Reviewed at `6d02d76c196aa0e52a06609f95b556aea48f7ba8` (2026-03-30). No end-to-end or component output is claimed here: the clone has unresolved gitlinks, no `.gitmodules`, no model weights/config, no input video, and no reference outputs.

In [1]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
pd.set_option("display.max_colwidth", 100)

manifest = pd.DataFrame([
    ("mmpose", "759b39c13fea6ba094afc1fa932f51dc1b11cbf9", "gitlink; empty directory; no .gitmodules mapping"),
    ("marker-enhancer", "3179260bbb4f21f8b114f059448def10bdd4443c", "gitlink; empty directory; no .gitmodules mapping"),
    ("model weights/config", "not present", "exact MMPose topology/checkpoint cannot be established"),
    ("sample input/output", "not present", "no reproducible run fixture"),
])
manifest.columns = ["dependency/artifact", "recorded value", "checkout result"]
display(manifest)

,dependency/artifact,recorded value,checkout result
0,mmpose,759b39c13fea6ba094afc1fa932f51dc1b11cbf9,gitlink; empty directory; no .gitmodules mapping
1,marker-enhancer,3179260bbb4f21f8b114f059448def10bdd4443c,gitlink; empty directory; no .gitmodules mapping
2,model weights/config,not present,exact MMPose topology/checkpoint cannot be established
3,sample input/output,not present,no reproducible run fixture


In [2]:
pipeline = pd.DataFrame([
    ("2-D/3-D pose", "MMPose container", "CUDA/NVIDIA hard-coded in Compose", "not executed"),
    ("marker enhancement", "claimed LSTM/Transformer", "source and weights absent", "not verifiable"),
    ("kinematics", "OpenSim LaiUhlrich2022_opencap", "CPU feasible if a valid TRC exists", "not executed"),
    ("events", "wrist-Z Savitzky–Golay heuristic", "code present", "source inspected"),
    ("metrics", "pelvis/lumbar OpenSim coordinates", "code present", "source inspected"),
], columns=["stage", "implementation", "hardware/reproducibility", "evidence status"])
display(pipeline)

,stage,implementation,hardware/reproducibility,evidence status
0,2-D/3-D pose,MMPose container,CUDA/NVIDIA hard-coded in Compose,not executed
1,marker enhancement,claimed LSTM/Transformer,source and weights absent,not verifiable
2,kinematics,OpenSim LaiUhlrich2022_opencap,CPU feasible if a valid TRC exists,not executed
3,events,wrist-Z Savitzky–Golay heuristic,code present,source inspected
4,metrics,pelvis/lumbar OpenSim coordinates,code present,source inspected


## Event and metric definitions found in source

Missing pose frames are forward-filled. The source fixes address to frame 0 and finish to the last frame, smooths wrist Z with a Savitzky–Golay window of 11, selects the first prominent wrist-Z peak (falling back to the maximum in the first 60%) as top, then the minimum after top and before 85% duration as impact. No probability or timing interval is returned.

From OpenSim `.mot`, pelvis rotation is `pelvis_rotation`, torso rotation is pelvis plus lumbar rotation, and X-factor is `abs(lumbar_rotation)`. Curves are zeroed at the first frame, sign-flip heuristics are applied, and configuration adds manual event offsets (top: pelvis −10°, torso −20°; impact: pelvis +10°, torso +15°). Those offsets mean displayed event values are adjusted outputs, not untouched inverse-kinematics measurements.

In [3]:
validation = pd.DataFrame([
    ("swing-event timing", "reliable", "project-authored qualitative summary only"),
    ("wrist tracking", "reliable", "project-authored qualitative summary only"),
    ("pelvis rotation / X-factor at top", "reliable", "project-authored qualitative summary only"),
    ("impact, low handicap", "good", "project-authored qualitative summary only"),
    ("impact, high handicap", "mixed", "project-authored qualitative summary only"),
    ("finish, angular velocity, sagittal tilt", "unreliable", "project-authored qualitative summary only"),
], columns=["claim", "repository wording", "independent status"])
display(validation)

hardware = pd.DataFrame([
    ("Current Compose path", "NVIDIA runtime + --device cuda", "No: host is Intel Arc"),
    ("MMPose in principle", "official CPU device option", "Potentially, but missing source/config/weights blocks test"),
    ("OpenSim", "CPU", "Potentially, if upstream TRC exists"),
    ("Intel XPU", "modern PyTorch supports Intel GPU packages", "Not supported by this pinned project stack"),
], columns=["path", "requirement", "this workstation"])
display(hardware)
print("EXECUTED_OK: Pose2Par source-review notebook")

,claim,repository wording,independent status
0,swing-event timing,reliable,project-authored qualitative summary only
1,wrist tracking,reliable,project-authored qualitative summary only
2,pelvis rotation / X-factor at top,reliable,project-authored qualitative summary only
3,"impact, low handicap",good,project-authored qualitative summary only
4,"impact, high handicap",mixed,project-authored qualitative summary only
5,"finish, angular velocity, sagittal tilt",unreliable,project-authored qualitative summary only


,path,requirement,this workstation
0,Current Compose path,NVIDIA runtime + --device cuda,No: host is Intel Arc
1,MMPose in principle,official CPU device option,"Potentially, but missing source/config/weights blocks test"
2,OpenSim,CPU,"Potentially, if upstream TRC exists"
3,Intel XPU,modern PyTorch supports Intel GPU packages,Not supported by this pinned project stack


EXECUTED_OK: Pose2Par source-review notebook


## Reuse decision

The stage boundaries, TRC/MOT interchange, and explicit reliability categories are valuable design references. The numerical offsets, validation labels, and exact pose/enhancer claims are not reusable evidence without the missing revisions, weights, fixtures, and an independent protocol. The top-level repository is MIT; the absent marker-enhancer source leaves its licence unresolved.